# V41: The Delta Booster (Residual Forecasting) 🚀
Notebook ini dirancang khusus untuk berjalan di **Google Colab**. Kita menggunakan teknik *Grandmaster* dengan mengubah target prediksi dari *Ketinggian Absolut* menjadi **Selisih terhadap Musim Tahun Lalu (Delta)**. Fitur Geospasial HydroRIVERS untuk **seluruh 30 Stasiun** juga disuntikkan secara utuh!

In [ ]:
# 1. MOUNT GOOGLE DRIVE (Wajib untuk Colab)
from google.colab import drive
import os

print("Meminta Izin Akses Google Drive...")
drive.mount('/content/drive')

# ==== UBAH BARIS INI SESUAI DENGAN LOKASI FOLDER ANDA DI GOOGLE DRIVE ====
# Contoh: '/content/drive/MyDrive/Kaggle/SSF-SSDS/'
PROJECT_PATH = '/content/drive/MyDrive/SSF-SSDS/'

# Pindah direktori agar pembacaan file lebih mudah
os.chdir(PROJECT_PATH)
print(f"Bekerja di direktori: {os.getcwd()}")

In [ ]:
# 2. IMPORT LIBRARY & LOAD DATA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import optuna
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import warnings
import json
warnings.filterwarnings('ignore')

print("Memuat Dataset Utama...")
train_df = pd.read_csv('train.csv', parse_dates=['datetime'])
test_df = pd.read_csv('test.csv')
env_df = pd.read_csv('data_pendukung/data_lingkungan.csv', parse_dates=['datetime'])
sample_sub = pd.read_csv('sample_submission.csv')

# Parse Test Datetime
test_parsed = test_df.copy()
split_info = test_parsed['id'].str.split(' - ', n=1, expand=True)
test_parsed['datetime'] = pd.to_datetime(split_info[0])
test_parsed['nama_pos'] = split_info[1]

# Memuat File JSON Geospasial HydroRIVERS secara utuh!
with open('hydro_mapping.json', 'r') as f:
    hydro_mapping = json.load(f)

print(f"Data Loaded: Train {train_df.shape}, Test {test_parsed.shape}, HydroRIVERS: {len(hydro_mapping)} stasiun.")

### Tahap 1: The Magic of Time-Shifting (Mencari Garis Dasar Musiman)
Kita menggabungkan *Train* dan *Test* untuk mencari nilai `tma_mdpl` yang persis terjadi 1 tahun (365 hari) yang lalu. Ini akan menjadi *Baseline* (Garis Dasar) kita.

In [ ]:
# Gabungkan Target Train dan Test untuk Time-Shifting
tma_history = train_df[['nama_pos', 'datetime', 'tma_mdpl']].copy()

# Buat kolom waktu "Tahun Lalu"
df_full = pd.concat([train_df, test_parsed], ignore_index=True)
df_full['datetime_minus_1y'] = df_full['datetime'] - pd.Timedelta(days=365)

# Siapkan data historis untuk di-Lookup
lookup_df = tma_history.rename(columns={'datetime': 'datetime_minus_1y', 'tma_mdpl': 'tma_lag_365d'})

# Lakukan Merge ke Masa Lalu
df_full = pd.merge(df_full, lookup_df, on=['nama_pos', 'datetime_minus_1y'], how='left')

# Jika persis 365 hari yang lalu tidak ada data, kita forward-fill / backward-fill per stasiun
df_full['tma_lag_365d'] = df_full.groupby('nama_pos')['tma_lag_365d'].transform(lambda x: x.ffill().bfill())

# --- THE DELTA TRANSFORMATION ---
# Inilah jantung dari V41. Mesin HANYA diajari menebak selisihnya!
df_full['target_delta'] = df_full['tma_mdpl'] - df_full['tma_lag_365d']

print("Target Delta dan Baseline 365 hari berhasil diekstrak.")

### Tahap 2: Feature Engineering Lingkungan & Injeksi Geospasial

In [ ]:
# Gabungkan dengan cuaca
env_df = env_df.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

# Hitung Fitur Fisika (Exponential Weighted) untuk mengukur serapan tanah
env_df['rain_ewm3'] = env_df.groupby('nama_pos')['rainfall_mm'].transform(lambda x: x.ewm(span=3, adjust=False).mean())
env_df['rain_ewm7'] = env_df.groupby('nama_pos')['rainfall_mm'].transform(lambda x: x.ewm(span=7, adjust=False).mean())

# Gabungkan cuaca ke df_full
df_full = pd.merge(df_full, env_df[['nama_pos', 'datetime', 'rainfall_mm', 'soil_moisture_0_7cm', 'temperature_c', 'rain_ewm3', 'rain_ewm7']], on=['nama_pos', 'datetime'], how='left')

# Injeksi Data Geospasial HydroRIVERS
df_full['UPLAND_SKM'] = df_full['nama_pos'].map(lambda x: hydro_mapping.get(x, {}).get('UPLAND_SKM', 0))
df_full['DIS_AV_CMS'] = df_full['nama_pos'].map(lambda x: hydro_mapping.get(x, {}).get('DIS_AV_CMS', 0))
df_full['ORD_STRA'] = df_full['nama_pos'].map(lambda x: hydro_mapping.get(x, {}).get('ORD_STRA', 1))

# Fitur Tanggal standar
df_full['month'] = df_full['datetime'].dt.month
df_full['dayofyear'] = df_full['datetime'].dt.dayofyear

print("Fitur Cuaca & Geospasial berhasil disuntikkan!")

### Tahap 3: Optuna XGBoost (Fokus pada Target Delta)
Kita tidak menggunakan `reg:tweedie` lagi karena target Delta bisa bernilai negatif (kemarau). Kita menggunakan `reg:squarederror`.

In [ ]:
optuna.logging.set_verbosity(optuna.logging.ERROR)

stations = df_full['nama_pos'].unique()
features = ['rainfall_mm', 'rain_ewm3', 'rain_ewm7', 'soil_moisture_0_7cm', 'temperature_c', 'month', 'dayofyear', 'UPLAND_SKM', 'DIS_AV_CMS', 'ORD_STRA', 'tma_lag_365d']

station_models = {}
station_scores = {}
delta_predictions = {}

N_TRIALS = 40
print(f"Mulai Melatih Model Delta untuk 30 Stasiun (Total {30 * N_TRIALS} Trials)...\n")

for st in stations:
    st_data = df_full[df_full['nama_pos'] == st].copy()
    
    # Pisahkan Train dan Test
    train_st = st_data[st_data['target_delta'].notna()].sort_values('datetime')
    test_st = st_data[st_data['tma_mdpl'].isna()].sort_values('datetime')
    
    X_train_st = train_st[features]
    y_train_st = train_st['target_delta']
    X_test_st = test_st[features]
    
    def objective(trial):
        params = {
            'objective': 'reg:squarederror', # WAJIB SQUAREDERROR KARENA DELTA BISA NEGATIF
            'eval_metric': 'rmse',
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'n_estimators': 300
        }
        
        tscv = TimeSeriesSplit(n_splits=3)
        scores = []
        for train_idx, val_idx in tscv.split(X_train_st):
            X_tr, X_va = X_train_st.iloc[train_idx], X_train_st.iloc[val_idx]
            y_tr, y_va = y_train_st.iloc[train_idx], y_train_st.iloc[val_idx]
            
            model = xgb.XGBRegressor(**params, random_state=42)
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            preds = model.predict(X_va)
            scores.append(np.sqrt(mean_squared_error(y_va, preds)))
            
        return np.mean(scores)
        
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=N_TRIALS)
    
    # Latih model final dengan seluruh data Train stasiun ini
    best_params = study.best_params
    best_params['objective'] = 'reg:squarederror'
    best_params['n_estimators'] = 300
    
    final_model = xgb.XGBRegressor(**best_params, random_state=42)
    final_model.fit(X_train_st, y_train_st)
    
    # PREDIKSI DELTA UNTUK TEST SET
    preds_delta_test = final_model.predict(X_test_st)
    
    # PREDIKSI DELTA UNTUK TRAIN SET (Untuk Visualisasi)
    preds_delta_train = final_model.predict(X_train_st)
    
    # REKONSTRUKSI ABSOLUT (Sihir Sebenarnya!)
    # Tebakan Mutlak = Prediksi Selisih + Ketinggian Tahun Lalu
    train_st['pred_absolute'] = preds_delta_train + train_st['tma_lag_365d']
    test_st['pred_absolute'] = preds_delta_test + test_st['tma_lag_365d']
    
    station_models[st] = final_model
    station_scores[st] = study.best_value
    
    # Simpan hasil untuk visualisasi dan submission
    delta_predictions[st] = {
        'train': train_st,
        'test': test_st
    }
    
    print(f"[{st}] - Best DELTA RMSE: {study.best_value:.4f}")

print(f"\n🏆 GRAND AVERAGE DELTA RMSE ACROSS 30 STATIONS: {np.mean(list(station_scores.values())):.4f}")

### Evaluasi Visual (Aktual vs Prediksi)
Apakah grafik kita masih mendatar pengecut? Mari kita buktikan dengan *Widget* Interaktif!

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

stations_list = sorted(list(stations))

def plot_reconstructed(station_name):
    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(15, 6))
    
    # Ambil data rekonstruksi mutlak
    df_train = delta_predictions[station_name]['train']
    
    # Plot 300 baris terakhir agar grafiknya terlihat jelas (tidak menumpuk)
    plot_df = df_train.tail(300)
    
    ax.plot(plot_df['datetime'], plot_df['tma_mdpl'], label='Aktual TMA (Meter)', color='blue', linewidth=2.5, alpha=0.7)
    ax.plot(plot_df['datetime'], plot_df['pred_absolute'], label='Prediksi TMA (V41 Delta)', color='red', linestyle='--', linewidth=2, alpha=0.9)
    ax.plot(plot_df['datetime'], plot_df['tma_lag_365d'], label='Garis Dasar (Musim Lalu)', color='green', linestyle=':', linewidth=1.5, alpha=0.5)
    
    ax.set_title(f"Visualisasi Absolute TMA (Rekonstruksi Delta) - [{station_name}]")
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    plt.show()

widgets.interactive(plot_reconstructed, station_name=widgets.Dropdown(options=stations_list, description='Stasiun:'))

### Submission File Generator
Memasukkan tebakan mutlak yang sudah direkonstruksi ke format CSV Kaggle.

In [ ]:
final_preds = []

for st in stations:
    df_test = delta_predictions[st]['test']
    
    # Buat format ID
    temp_df = pd.DataFrame({
        'id': df_test['datetime'].astype(str) + ' - ' + st,
        'tma_mdpl': df_test['pred_absolute']
    })
    final_preds.append(temp_df)

sub_pred = pd.concat(final_preds, ignore_index=True)

# Merge dengan sample_submission untuk memastikan urutannya tepat
sub = pd.merge(sample_sub[['id']], sub_pred, on='id', how='left')

# Amankan data jika ada NaN (misal 365 hari lalu data kosong total)
sub['tma_mdpl'] = sub['tma_mdpl'].fillna(method='ffill').fillna(method='bfill')

output_file = 'submission_V41_delta_booster.csv'
sub.to_csv(output_file, index=False)
print(f"✅ File '{output_file}' berhasil disimpan di Google Drive Anda! Siap disubmit ke Kaggle!")